<a href="https://colab.research.google.com/github/Imran-Nayem/Machine_Learning/blob/main/Copy_of_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# load data
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

print(df.head())
print("Shape:", df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# load data
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# drop irrelevant columns
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
# features & target
X = df.drop("Survived", axis=1)
y = df["Survived"]

#types of column
num_cols = ["Age", "Fare", "SibSp", "Parch"]
cat_cols = ["Sex", "Embarked", "Pclass"]

#pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:

from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

##ans-4
random Forest selected cause it performs well on tabular datasets, handles non-linear relationships, reduces overfitting  and needs minimal feature scaling compared with the other models.

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("Cross-validation scores:", scores)
print("Mean:", np.mean(scores))
print("Std:", np.std(scores))

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
import gradio as gr
import joblib
import pandas as pd

model = joblib.load("model.pkl")

def predict(pclass, sex, age, sibsp, parch, fare, embarked):
    data = pd.DataFrame([{
        "Pclass": pclass,
        "Sex": sex,
        "Age": age,
        "SibSp": sibsp,
        "Parch": parch,
        "Fare": fare,
        "Embarked": embarked
    }])

    prediction = model.predict(data)[0]
    return "Survived" if prediction == 1 else "Did Not Survive"

interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Pclass"),
        gr.Radio(["male", "female"], label="Sex"),
        gr.Number(label="Age"),
        gr.Number(label="SibSp"),
        gr.Number(label="Parch"),
        gr.Number(label="Fare"),
        gr.Radio(["C", "Q", "S"], label="Embarked")
    ],
    outputs="text"
)

interface.launch()

In [ ]:
import joblib
joblib.dump(best_model, "model.pkl")

In [ ]:
import os
print(os.listdir())

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Imran-Nayem/Titanic-Passanger-survial-Prediction"

api.upload_file(
    path_or_fileobj="model.pkl",
    path_in_repo="model.pkl",
    repo_id=repo_id,
    repo_type="space"
)

In [ ]:
app_code = """
import gradio as gr
import joblib
import pandas as pd

model = joblib.load("model.pkl")

def predict(pclass, sex, age, sibsp, parch, fare, embarked):
    data = pd.DataFrame([{
        "Pclass": int(pclass),
        "Sex": sex,
        "Age": float(age),
        "SibSp": int(sibsp),
        "Parch": int(parch),
        "Fare": float(fare),
        "Embarked": embarked
    }])

    try:
        pred = model.predict(data)[0]
        return "Survived" if pred == 1 else "Did Not Survive"
    except Exception as e:
        return "Error: " + str(e)

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Pclass", value=3),
        gr.Radio(["male", "female"], label="Sex", value="male"),
        gr.Number(label="Age", value=25),
        gr.Number(label="SibSp", value=0),
        gr.Number(label="Parch", value=0),
        gr.Number(label="Fare", value=7.25),
        gr.Radio(["C", "Q", "S"], label="Embarked", value="S")
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="Titanic Passenger Survival Prediction"
)

demo.launch()
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py updated ")

In [ ]:
req = """
pandas
numpy
scikit-learn==1.6.1
gradio
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(req)

print("requirements.txt fixed ")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Imran-Nayem/Titanic-Passanger-survial-Prediction"

api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=repo_id,
    repo_type="space"
)

api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=repo_id,
    repo_type="space"
)